# Projeto de Machine Learning — Previsão de consumo de combustível (MPG)

**Objetivo:** construir e avaliar modelos de regressão para prever `mpg` (miles per gallon) a partir das características numéricas dos carros.

Este notebook reúne, em uma sequência organizada, os conceitos estudados:
1. Carregamento e inspeção dos dados.
2. Separação entre variáveis explicativas (`X`) e variável-alvo (`y`).
3. Separação treino/teste.
4. Pipeline de preparação e regressão linear.
5. Avaliação com métricas de regressão.
6. Validação cruzada.
7. Comparação entre Ridge, Lasso e ElasticNet usando diferentes valores de `alpha`.
8. Inspeção dos coeficientes do Lasso.

> **Nota sobre o escopo:** o arquivo original enviado contém o projeto MPG. Ele não inclui todos os notebooks anteriores de Titanic, Diamonds ou House Prices. Por isso, este documento organiza o conteúdo disponível, sem inventar etapas que não estavam no arquivo original.


## 1. Importar bibliotecas

- `pandas`: tabelas e manipulação de dados.
- `numpy`: operações numéricas.
- `seaborn`: carregamento do conjunto de dados de exemplo.
- `scikit-learn`: divisão dos dados, preparação, modelos e métricas.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

# Semente usada para tornar a divisão reproduzível.
RANDOM_STATE = 42


## 2. Carregar e conhecer os dados

O conjunto `mpg` contém informações de carros e o consumo de combustível. Primeiro, observamos algumas linhas, os tipos de dados e os valores ausentes.

Se o ambiente não conseguir baixar o conjunto do Seaborn, será necessário disponibilizar o arquivo CSV localmente e carregá-lo com `pd.read_csv()`.


In [ ]:
df = sns.load_dataset("mpg")

display(df.head())

print("Dimensões (linhas, colunas):", df.shape)
print("\nTipos e valores não ausentes:")
df.info()

print("\nValores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False))


## 3. Definir as variáveis

- `y`: variável-alvo que queremos prever, `mpg`.
- `X`: variáveis usadas para fazer a previsão.

Neste primeiro modelo, removemos `origin` e `name` para trabalhar com as variáveis numéricas. Isso é uma escolha didática: informações categóricas podem ser aproveitadas em outro experimento usando codificação apropriada.


In [ ]:
# A variável-alvo é o consumo de combustível.
y = df["mpg"]

# Retiramos o alvo e as colunas categóricas que não serão usadas neste experimento.
X = df.drop(columns=["mpg", "origin", "name"])

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)
display(X.head())


## 4. Separar treino e teste

O conjunto de treino serve para ajustar o modelo. O conjunto de teste serve para avaliar o desempenho em observações que não foram usadas diretamente no ajuste.

Usamos 75% dos dados para treino e 25% para teste. `random_state=42` torna a divisão reproduzível.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print("Treino:", X_train.shape, y_train.shape)
print("Teste:", X_test.shape, y_test.shape)


## 5. Criar um pipeline de Regressão Linear

O pipeline reúne a preparação dos dados e o modelo em uma única sequência:

1. `SimpleImputer(strategy="median")`: substitui valores numéricos ausentes pela mediana calculada nos dados usados para ajustar cada etapa.
2. `StandardScaler()`: padroniza as variáveis para média próxima de zero e desvio padrão próximo de um.
3. `LinearRegression()`: aprende a relação entre as variáveis e o consumo.

O pipeline ajuda a evitar vazamento de dados: durante a validação cruzada, o imputador e o escalonador são ajustados separadamente dentro de cada partição de treino.


In [ ]:
pipeline_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

# Ajustar o pipeline somente com os dados de treino.
pipeline_linear.fit(X_train, y_train)


## 6. Fazer previsões e avaliar o modelo

Calculamos previsões tanto no treino quanto no teste para observar o ajuste e a capacidade de generalização.

### Métricas
- **MAE:** erro absoluto médio; menor é melhor.
- **MSE:** erro quadrático médio; penaliza mais os erros grandes.
- **RMSE:** raiz do MSE; fica na mesma unidade da variável-alvo.
- **MAPE:** erro percentual absoluto médio; pode ser difícil de interpretar quando os valores reais são próximos de zero.
- **R²:** compara o modelo com uma referência que prevê a média; em geral, quanto maior, melhor.


In [ ]:
# Previsões do modelo ajustado.
y_pred_train = pipeline_linear.predict(X_train)
y_pred_test = pipeline_linear.predict(X_test)


def calcular_metricas(y_real, y_previsto):
    """Retorna as principais métricas para um problema de regressão."""
    mse = mean_squared_error(y_real, y_previsto)

    return {
        "MAE": mean_absolute_error(y_real, y_previsto),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAPE": mean_absolute_percentage_error(y_real, y_previsto),
        "R2": r2_score(y_real, y_previsto),
    }


metricas_treino = calcular_metricas(y_train, y_pred_train)
metricas_teste = calcular_metricas(y_test, y_pred_test)

comparacao_metricas = pd.DataFrame({
    "Treino": metricas_treino,
    "Teste": metricas_teste
})

display(comparacao_metricas.round(4))


### Como interpretar as métricas

Não basta olhar para uma métrica isolada. Compare treino e teste:
- Se o desempenho no treino for muito melhor que no teste, investigue possível overfitting.
- Se ambos forem ruins, o modelo pode estar subajustado ou as variáveis podem não conter informação suficiente.
- Resultados semelhantes entre treino e teste são úteis, mas não garantem por si só que o modelo seja bom.


## 7. Validação cruzada da Regressão Linear

A validação cruzada divide o conjunto de treino em cinco partes (*folds*). Em cada rodada, uma parte valida o modelo e as outras partes o ajustam.

Usamos R² como métrica nesta seção. A média resume o desempenho observado e o desvio padrão indica a variação entre as partições.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores_r2 = cross_val_score(
    pipeline_linear,
    X_train,
    y_train,
    cv=cv,
    scoring="r2"
)

print("R² por fold:", np.round(scores_r2, 4))
print(f"R² médio: {scores_r2.mean():.4f}")
print(f"Desvio padrão do R²: {scores_r2.std():.4f}")


## 8. Comparar Ridge, Lasso e ElasticNet

Os três modelos aplicam regularização:
- **Ridge (L2):** reduz os coeficientes; normalmente não os zera.
- **Lasso (L1):** pode zerar coeficientes, funcionando como uma forma de seleção de variáveis.
- **ElasticNet (L1 + L2):** combina as duas penalizações.

`alpha` controla a força da regularização. Neste experimento, testamos cinco valores. No ElasticNet, mantemos `l1_ratio=0.5` para focar a comparação no efeito de `alpha`.

A mesma estratégia de validação cruzada é usada para todas as combinações, e a escala e imputação acontecem dentro do pipeline.


In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]

resultados = []

for alpha in alphas:
    modelos = {
        "Ridge": Ridge(alpha=alpha),
        "Lasso": Lasso(alpha=alpha, max_iter=10000),
        "ElasticNet": ElasticNet(
            alpha=alpha,
            l1_ratio=0.5,
            max_iter=10000
        )
    }

    for nome_modelo, algoritmo in modelos.items():
        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", algoritmo)
        ])

        # Scikit-learn retorna scores negativos para neg_mean_absolute_error.
        # Multiplicamos a média por -1 para apresentar o MAE em valor positivo.
        scores_mae_negativos = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring="neg_mean_absolute_error"
        )

        resultados.append({
            "Modelo": nome_modelo,
            "Alpha": alpha,
            "MAE médio": -scores_mae_negativos.mean(),
            "Desvio MAE": scores_mae_negativos.std()
        })

resultado_df = pd.DataFrame(resultados)

display(
    resultado_df
    .sort_values("MAE médio")
    .reset_index(drop=True)
    .round(4)
)


### Interpretar a tabela de regularização

- Menor **MAE médio** indica menor erro médio nas partições de validação.
- Menor **Desvio MAE** indica menor variação observada entre as partições.
- Um `alpha` muito alto pode penalizar demais os coeficientes e aumentar o erro, sinalizando possível underfitting.
- Se as diferenças de MAE forem pequenas, evite declarar um vencedor definitivo sem considerar a incerteza e o desempenho em dados novos.

**Resultado observado na execução anterior deste projeto:** Lasso com `alpha=0.1` obteve MAE médio de aproximadamente `2.7214` e desvio padrão de `0.2166`. Os resultados podem variar conforme a versão das bibliotecas e os dados usados.


## 9. Inspecionar os coeficientes do Lasso

Vamos ajustar o Lasso com `alpha=0.1`, a melhor combinação observada na busca anterior, e examinar os coeficientes.

Um coeficiente igual a zero significa que aquela variável não participa diretamente da equação ajustada pelo Lasso. Isso não prova que a variável seja inútil em todos os modelos ou conjuntos de dados.


In [ ]:
alpha_escolhido = 0.1

lasso_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=alpha_escolhido, max_iter=10000))
])

lasso_pipeline.fit(X_train, y_train)

coeficientes = pd.Series(
    lasso_pipeline.named_steps["model"].coef_,
    index=X_train.columns,
    name="Coeficiente"
).sort_values()

display(coeficientes.to_frame())

variaveis_zeradas = coeficientes[coeficientes == 0].index.tolist()
print("Quantidade de coeficientes zerados:", len(variaveis_zeradas))
print("Variáveis com coeficiente zero:", variaveis_zeradas)


## 10. Conclusão e próximos passos

Ao concluir este notebook, você terá praticado:
- separação de variáveis e divisão treino/teste;
- imputação e escalonamento dentro de um pipeline;
- avaliação por MAE, MSE, RMSE, MAPE e R²;
- validação cruzada;
- comparação de modelos e hiperparâmetros;
- inspeção de coeficientes do Lasso.

### Limitações deste experimento
1. `origin` e `name` foram excluídas; outros experimentos podem testar codificação de variáveis categóricas e engenharia de atributos.
2. A busca de hiperparâmetros usa uma grade pequena e `l1_ratio=0.5` fixo para ElasticNet.
3. O conjunto de teste foi utilizado na avaliação inicial da regressão linear no notebook original. Para um relatório final rigoroso após muitas decisões de modelagem, mantenha um teste final realmente reservado ou use uma estratégia de validação mais completa.

### Sugestão de estrutura para o GitHub

```text
projeto-mpg-machine-learning/
├── Projeto_MPG_ML_Organizado.ipynb
├── README.md
└── requirements.txt
```

Para organizar os outros estudos (Titanic, Diamonds, House Prices, etc.), precisaremos reunir os respectivos notebooks ou arquivos. Este documento organiza apenas o conteúdo MPG disponível no arquivo enviado.
